# Baseline Linear Regression for HDB Rental Prices

This notebook performs basic cleaning on the raw training data, creates a reproducible train/test split, trains a linear regression model, and evaluates its predictions.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'data' / 'train.csv').exists()), HERE)
if not (PROJECT_ROOT / 'data' / 'train.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'
df = pd.read_csv(DATA_PATH)
print(f'Loaded {DATA_PATH}')
print('Shape:', df.shape)
df.head()

Loaded /Users/marklim/Masters/CS5228/CS5228-project/data/train.csv
Shape: (150000, 11)


,RENT_APPROVAL_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,FURNISHED,LEASE_COMMENCE_DATE,FEE,MONTHLY_RENT
0,2021-01,punggol,604a,punggol road,4 room,model a,93.0,yes,2012,0.0,1800
1,2021-01,sengkang,290c,compassvale crescent,5 room,improved,110.0,yes,2001,0.0,2000
2,2021-01,queenstown,5,ghim moh road,3 room,improved,65.0,yes,1976,0.0,2000
3,2021-01,sengkang,436b,fernvale road,4 room,model a,96.0,yes,2010,0.0,1400
4,2021-01,sembawang,467b,admiralty drive,4 room,premium apartment,102.0,yes,2001,0.0,1800


## Basic cleaning

The target is `MONTHLY_RENT`. Row identifiers and the constant `FEE` column are excluded, while `FURNISHED` is retained and encoded as binary. Text columns are normalized and known category variants are canonicalized. The approval month is converted into year, month, and quarter features. Missing numeric values are median-imputed and missing categorical values are treated as an explicit category inside the modeling pipeline.

In [5]:
TARGET = 'MONTHLY_RENT'
work = df.copy()

# Remove exact duplicate rows from the baseline dataset.
work = work.drop_duplicates().reset_index(drop=True)

# Normalize all text fields: trim whitespace, collapse repeated spaces, and lowercase.
text_columns = work.select_dtypes(include=['object', 'string']).columns
for column in text_columns:
    work[column] = (work[column].astype('string').str.strip()
                    .str.replace(r'\s+', ' ', regex=True).str.lower())

# Standardize common street-type spellings and abbreviations.
street_replacements = {
    r'\bavenue\b': 'ave', r'\broad\b': 'rd',
    r'\bstreet\b': 'st', r'\bdrive\b': 'dr',
    r'\bcentral\b': 'ctrl', r'\bplace\b': 'pl',
    r'\bclose\b': 'cl', r'\bcrescent\b': 'cres',
    r'\blane\b': 'ln', r'\blink\b': 'link',
    r'\bparkway\b': 'pkwy', r'\bterrace\b': 'ter',
    r'\bview\b': 'vw', r'\bwalk\b': 'walk',
}
for pattern, replacement in street_replacements.items():
    work['STREET'] = work['STREET'].str.replace(pattern, replacement, regex=True)
work['STREET'] = work['STREET'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Canonicalize known category variants. These mappings make spelling, case, and
# formatting differences resolve to one category before one-hot encoding.
work['FLAT_TYPE'] = work['FLAT_TYPE'].replace({
    '1 room': '1-room', '2 room': '2-room', '3 room': '3-room',
    '4 room': '4-room', '5 room': '5-room',
})
work['FURNISHED'] = work['FURNISHED'].replace({
    'yes': 1, 'y': 1, 'true': 1, '1': 1,
    'no': 0, 'n': 0, 'false': 0, '0': 0,
}).astype('Int64')

approval_date = pd.to_datetime(work.pop('RENT_APPROVAL_DATE'), format='%Y-%m', errors='coerce')
work['RENT_YEAR'] = approval_date.dt.year
work['RENT_MONTH'] = approval_date.dt.month
work['RENT_QUARTER'] = approval_date.dt.quarter

y = pd.to_numeric(work.pop(TARGET), errors='coerce')
work = work.drop(columns=['Id', 'ROW_ID', 'FEE'], errors='ignore')
valid_target = y.notna()
X = work.loc[valid_target].copy()
y = y.loc[valid_target].copy()

# Treat blank strings as missing values.
X = X.replace(r'^\s*$', np.nan, regex=True)
print('Rows after cleaning:', len(X))
print('Features:', X.shape[1])
print('Target summary:')
y.describe()

Rows after cleaning: 149575
Features: 11
Target summary:


count    149575.000000
mean       2717.921671
std         749.964607
min         300.000000
25%        2100.000000
50%        2650.000000
75%        3200.000000
max        7600.000000
Name: MONTHLY_RENT, dtype: float64

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

Train rows: 119,660
Test rows: 29,915
Numeric features: 6
Categorical features: 5


In [7]:
print(numeric_features)
print(categorical_features)

['FLOOR_AREA_SQM', 'FURNISHED', 'LEASE_COMMENCE_DATE', 'RENT_YEAR', 'RENT_MONTH', 'RENT_QUARTER']
['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']


## Evaluation

In [11]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))
metrics = pd.DataFrame({
    'metric': ['MAE', 'RMSE', 'R²'],
    'value': [
        mean_absolute_error(y_test, predictions),
        rmse,
        r2_score(y_test, predictions),
    ],
})
metrics

,metric,value
0,MAE,388.692194
1,RMSE,508.771624
2,R²,0.543820


In [12]:
evaluation = pd.DataFrame({
    'actual': y_test.to_numpy(),
    'predicted': predictions,
})
evaluation['residual'] = evaluation['actual'] - evaluation['predicted']
print('Prediction summary:')
print(evaluation.describe())
print('Mean residual:', evaluation['residual'].mean())
evaluation.head(10)

Prediction summary:
             actual     predicted      residual
count  29915.000000  29915.000000  29915.000000
mean    2726.671770   2718.887616      7.784154
std      753.289904    561.516788    508.720575
min      450.000000    797.631477  -2862.425011
25%     2100.000000   2321.147235   -298.137264
50%     2700.000000   2702.488835     -6.217165
75%     3250.000000   3102.490426    319.946684
max     6600.000000   5210.419568   3138.697369
Mean residual: 7.784154248751001


,actual,predicted,residual
0,3300,2570.274529,729.725471
1,3700,3711.218621,-11.218621
2,3100,2206.630199,893.369801
3,3200,2682.074778,517.925222
4,3300,3302.175641,-2.175641
5,1770,1753.929848,16.070152
6,4200,3575.010086,624.989914
7,2600,2705.316309,-105.316309
8,2200,2217.176588,-17.176588
9,2200,3365.334376,-1165.334376
